# 01 · Exploratory data analysis (HAM10000)

## Summary of findings

- **Size:** 10,015 images of 7,470 lesions (1.34 images per lesion). Melanoma has the most repeats (1.81 per lesion), so it has fewer independent examples than its image count suggests.
- **Class imbalance:** nv is 66.9% of images, df only 1.1% (58x). Always predicting nv gives about 67% accuracy, so the main metric is balanced accuracy / macro-F1, plus per-class recall (especially melanoma) and the confusion matrix. Training needs class weights or weighted sampling.
- **Split:** grouped by lesion, about 70/15/15 per class, with no lesion in more than one split. Rare classes are tiny in val/test (df: 10/11 lesions, vasc: 14/14), so their scores are noisy. Select epochs by val loss and look at the test set only once.
- **Label quality and selection bias:** mel, bcc and akiec are 100% histopathology; nv is 55% follow-up only. Only suspicious moles get biopsied, so the dataset has fewer "hard" benign moles than a real clinic and scores will be optimistic.
- **Source shortcut risk:** vidir_molemax is 94% nv; its images look zoomed in, some have dark corners. 90% of akiec comes from rosendahl. Flips and rotations do not remove this. Mitigate with colour jitter and random resized crops; evaluate per source, check Grad-CAM, and consider leave-one-source-out testing.
- **Age confound:** median age is 45 for nv, 60 for mel and 70 for bcc/akiec. Skin visibly ages, so an image-only model can learn age instead of the lesion. Report melanoma recall by age group.
- **Missing data in three forms:** NaN (57 ages), "unknown" (57 sex, 234 localization) and 0 (some ages). Localization labels overlap between sources (acral vs foot/hand, trunk vs back/chest/abdomen). Ages are rounded to 5 years and capped at 85.
- **Images:** all 600x450. Hair appears in every source (noise rather than a shortcut). Preprocessing plan: resize the short side to 256, then crop 224x224 (random crop for training, centre crop for val/test).


In [ ]:
from pathlib import Path

import pandas as pd

from skinlesion.labels import CLASSES

DATA = Path("../data/raw")
meta = pd.read_csv(DATA / "HAM10000_metadata.csv")

print(meta.shape)
meta.head(20)

In [ ]:
counts = pd.DataFrame(
    {
        "images": meta.groupby("dx")["image_id"].count(),
        "lesions": meta.groupby("dx")["lesion_id"].nunique(),
    }
)
counts["imgs_per_lesion"] = (counts["images"] / counts["lesions"]).round(2)
counts["pct_images"] = (100 * counts["images"] / counts["images"].sum()).round(1)
counts["name"] = counts.index.map(CLASSES)

counts.sort_values("images", ascending=False)

In [ ]:
splits = pd.read_csv("../data/splits.csv")

lesion_splits = splits.groupby("lesion_id")["split"].nunique()
print("Lesions found in more than one split:", (lesion_splits > 1).sum())

pd.crosstab(splits["dx"], splits["split"], values=splits["lesion_id"], aggfunc="nunique")

In [ ]:
pd.crosstab(meta["dx"], meta["dx_type"], normalize="index").mul(100).round(1)

In [ ]:
src = pd.crosstab(meta["dataset"], meta["dx"])
src

In [ ]:
pd.crosstab(meta["dataset"], meta["dx"], normalize="index").mul(100).round(1)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

from skinlesion.labels import CLASS_CODES

IMG_DIR = DATA / "images"
N = 5

fig, axes = plt.subplots(len(CLASS_CODES), N, figsize=(2 * N, 2 * len(CLASS_CODES)))
for row, code in enumerate(CLASS_CODES):
    sample = meta[meta["dx"] == code].sample(N, random_state=0)
    for col, (_, r) in enumerate(sample.iterrows()):
        ax = axes[row, col]
        ax.imshow(Image.open(IMG_DIR / f"{r['image_id']}.jpg"))
        ax.set_title(r["dataset"], fontsize=7)
        ax.axis("off")
    axes[row, 0].text(
        -0.1, 0.5, code, transform=axes[row, 0].transAxes, ha="right", va="center", fontsize=11
    )
plt.tight_layout()
plt.show()

In [ ]:
sizes = pd.Series(
    [Image.open(IMG_DIR / f"{i}.jpg").size for i in meta["image_id"]],
    index=meta["image_id"],
)
sizes.value_counts()

In [ ]:
print(meta.isna().sum(), "\n")
for col in ["sex", "localization"]:
    print(meta[col].value_counts(), "\n")

In [ ]:
print(meta.isna().sum(), "\n")
for col in ["sex", "localization"]:
    print(meta[col].value_counts(), "\n")

In [ ]:
meta.groupby("dx")["age"].describe().round(1)